# Demo 10 — Outlier removal (before and after fitting)

Influential outliers can dominate a fit, so removing them on a principled basis —
and seeing how much they mattered — is part of a careful analysis. This demo shows
that on the `stackloss` dataset (R base; Brownlee 1965): the percentage of ammonia
lost during an industrial oxidation process over 21 plant runs, where observations
1, 3, 4, and 21 are textbook influential outliers.

The same linear model is fitted twice — once untouched, once after pre-fit IQR and
post-fit residual outlier removal:

    stack.loss ~ Air.Flow + Water.Temp + Acid.Conc.

Air.Flow is binned into three operating regimes for the plot, while Water.Temp and
Acid.Conc. enter as numeric covariates. Comparing the two runs shows how
kbstatpy's two-pass, principled outlier handling shifts the estimates, while
keeping the excluded points visible and the analysis valid under the resulting
imbalance.

## Run on Google Colab

On [Google Colab](https://colab.research.google.com)? Run the cell below first —
it installs kbstatpy, its R packages, and the demo data (~3–5 min the first time).
It is a no-op when you run this notebook locally from the kbstatpy source tree.
Then run the cells below to see the tables and figures rendered inline.

In [ ]:
# Google Colab only: install kbstatpy + its R packages + the demo data.
# (Does nothing when the notebook runs locally from the source tree.)
import sys
if 'google.colab' in sys.modules:
    !curl -sSL https://raw.githubusercontent.com/kimbostroem/kbstatpy/master/demos/colab_setup.sh | bash

## Setup

In [ ]:
import os

import matplotlib.pyplot as plt
from kbstatpy import Kbstat, KbstatOptions

## Options

The same model is run twice — once without outlier removal, once with both pre- and post-fit removal — reusing the same `options`, reconfigured in place.

In [ ]:
options = KbstatOptions()
options.in_file   = os.path.join(options.demo_dir, 'data/stackloss.csv')
options.out_dir   = ''   # display only; see the Save cell
options.y         = 'stack.loss'
options.y_units   = '%'
options.x         = 'Air.Flow'
options.covariate = 'Water.Temp, Acid.Conc.'
options.rename    = ('stack.loss -> StackLoss; Air.Flow -> AirFlow; '
                     'Water.Temp -> WaterTemp; Acid.Conc. -> AcidConc')

## Run 1 — No outlier removal

Files are written only if `out_dir` is set (empty here, so results are shown inline).

In [ ]:
kb_default = Kbstat(options)
kb_default.run();

## Run 2 — Pre-fit IQR + post-fit residual removal

Files are written only if `out_dir` is set (empty here, so results are shown inline).

In [ ]:
options.remove_outliers_prefit  = True
options.remove_outliers_postfit = True
kb_clean = Kbstat(options)
kb_clean.run();

## Save results

Everything above is shown inline. To also write both runs to disk, uncomment the lines below and run this cell — they set each run's `out_dir` and call `save()` on the results already computed (no re-run).

In [ ]:
# To write both runs to disk, uncomment (each saves to its own folder):
# kb_default.options.out_dir = 'results/demo_10_outliers/default'; kb_default.save()
# kb_clean.options.out_dir   = 'results/demo_10_outliers/clean';   kb_clean.save()
# kb_default.download_link('results/demo_10_outliers')   # remote server: one zip with both runs

## Interpretation

- Observations 1, 3, 4, and 21 are documented outliers (Brownlee, 1965) — visible as Cook's D spikes in the diagnostic plots.
- Compare the post-hoc tables from both runs: the clean model should show tighter confidence intervals and potentially different significance.
- Use Cook's D in the diagnostics to identify which observations were removed and whether their removal is scientifically justified.